# 05: ABAC Policies with Governed Tags

**Exam objective:** Understand Unity Catalog ABAC policies to centrally 
control row-level filtering and column masking for sensitive data.

**Free Edition note:** Tag UI is present in Catalog Explorer; Previews 
panel is absent in Settings. Some ABAC operations may be restricted. 
Where blocked, the syntax and concept are documented in markdown cells 
and the behavior is reasoned about rather than demonstrated.

**Conceptual relationship to exercises 03 and 04:** Per-table masks and 
filters (exercises 03 and 04) are object-level governance. ABAC is the 
modern alternative — tag-based, centrally managed, owner-override-proof. 
Databricks now recommends ABAC for most use cases.

In [0]:
USE CATALOG certprep;
USE SCHEMA governance;

In [0]:
DROP TABLE IF EXISTS sales_data;

CREATE TABLE sales_data (
  sale_id INT,
  region STRING,
  amount DECIMAL(10,2),
  customer_email STRING,
  customer_ssn STRING,
  sale_date DATE
);

INSERT INTO sales_data VALUES
  (1, 'North', 1250.00, 'alice@example.com', '123-45-6789', '2026-01-15'),
  (2, 'South', 890.50,  'bob@example.com',   '234-56-7890', '2026-01-16'),
  (3, 'East',  2100.75, 'carol@example.com', '345-67-8901', '2026-01-17'),
  (4, 'West',  1575.25, 'dan@example.com',   '456-78-9012', '2026-01-18');

SELECT * FROM sales_data;

## Add Governed Tags Via Catalog Explorer

Governed tags can be added via the Catalog Explorer UI. Governed tags can be applied at different levels: catalog, schema, table, column, etc. There are intuitive predefined system tags, denoted as `class.<type>`, or you can create a user defined tag and set its acceptable values. Principles can be granted permissions to manage or assign governed tags. 

Whether predefined or user-defined, applying a governed tag via the UI enforces the accepted values via a dropdown menu. 

- Tag policy: defines the tag's vocabulary and who can manage it.
- ABAC policy: uses the tag to apply filtering or masking.

See `system.information_schema.` for tag views for each level. ABAC Policies can match at any level.

In [0]:
-- Apply a tag to the table (this is the syntax; may or may not work on Free Edition)
ALTER TABLE sales_data SET TAGS ('sensitivity' = 'high');

In [0]:
-- Verify the tag is applied
SELECT * FROM system.information_schema.column_tags 
WHERE catalog_name = 'certprep' 
  AND schema_name = 'governance' 
  AND table_name = 'sales_data';

In [0]:
-- Table-level tags
SELECT * FROM system.information_schema.table_tags
WHERE catalog_name = 'certprep'
  AND schema_name = 'governance'
  AND table_name = 'sales_data';

In [0]:
-- Reuse the masking pattern from exercise 03
CREATE OR REPLACE FUNCTION abac_mask_ssn(ssn STRING)
RETURN
  CASE
    WHEN is_account_group_member('analysts') THEN ssn
    ELSE CONCAT('XXX-XX-', RIGHT(ssn, 4))
  END;

In [0]:
CREATE OR REPLACE POLICY mask_pii_columns
ON CATALOG certprep
COMMENT 'Masks any column tagged sensitivity=pii for non-analysts'
COLUMN MASK abac_mask_ssn
TO `account users`
FOR TABLES
MATCH COLUMNS has_tag_value('sensitivity', 'pii') AS pii_col
ON COLUMN pii_col;

In [0]:
SELECT * FROM sales_data;

In [0]:
-- Create a second tagged column on a different (new) table.
-- Watch the policy auto-apply.
CREATE TABLE certprep.governance.customer_records (
  customer_id INT,
  customer_ssn STRING,
  full_name STRING
);

INSERT INTO certprep.governance.customer_records VALUES
  (101, '987-65-4321', 'Alice Smith'),
  (102, '876-54-3210', 'Bob Jones');

-- Tag the new SSN column
ALTER TABLE certprep.governance.customer_records 
ALTER COLUMN customer_ssn 
SET TAGS ('sensitivity' = 'pii');

-- Query it
SELECT * FROM certprep.governance.customer_records;

In [0]:
SHOW POLICIES ON CATALOG certprep;

In [0]:
-- Drop the policy
DROP POLICY mask_pii_columns ON CATALOG certprep;

In [0]:
-- Untag the columns
ALTER TABLE certprep.governance.sales_data 
ALTER COLUMN customer_ssn 
UNSET TAGS ('sensitivity');

ALTER TABLE certprep.governance.customer_records 
ALTER COLUMN customer_ssn 
UNSET TAGS ('sensitivity');

-- Untag the table
ALTER TABLE certprep.governance.sales_data 
UNSET TAGS ('sensitivity');

In [0]:
-- Drop the demo table
DROP TABLE IF EXISTS certprep.governance.customer_records;

-- Drop the UDF
DROP FUNCTION IF EXISTS abac_mask_ssn;

In [0]:
-- Verify cleanup
SHOW POLICIES ON CATALOG certprep;
SELECT * FROM system.information_schema.column_tags 
WHERE catalog_name = 'certprep';
SHOW FUNCTIONS IN certprep.governance;

## Self Check Questions
1. Why are ABAC policies considered more secure than per-table column masks, beyond just the scaling advantage? Be specific about what an attacker or careless owner could do with a per-table mask that they couldn't do with an ABAC policy.
2. A team adds a new table to a catalog that has an ABAC column-mask policy in place. The new table has a column containing phone numbers. How does the column become masked? What's the team's responsibility, and what's automatic?
3. ABAC uses "governed tags" rather than free-form tags. Why does the distinction matter? What goes wrong if you let any user apply any tag value they want?
4. You define an ABAC row-filter policy at the catalog level that hides EU customer rows from a US analytics group. A table owner later attaches a per-table row filter to one of the tables that contradicts the policy (say, allows the US group to see EU rows). What happens — does the table-level filter override the ABAC policy, the other way around, or neither? Why?
5. ABAC policies require the principal to be a member of a group that the policy applies to (or specifically exempted). What's the implication for how you'd structure groups in a real org if you wanted to use ABAC heavily? Is there a tradeoff?

## Self Check Answers
1. ABAC policies are more secure because they sit at a higher level of governance than table owner privileges. Table owners have the ability to drop per-table masks, exposing the underlying data to anyone with SELECT privileges. ABAC policies on the other hand, can only be modified by users with policy-management privileges at the catalog or schema level, which typically a much smaller group of users.

2. The new table inherits the ABAC policy from its parent catalog. If the catalog policy handles phone number masking, the team must tag the relevant column so it is visible to the policy. The policy will then be able to see the column and mask it automatically. 

One small refinement worth noting: there's a third layer — automated data classification — that can reduce even the tagging responsibility, because Databricks can scan columns and apply class.* tags automatically (the System tags you saw in the governed tags list). So in a mature setup, tagging itself becomes increasingly automatic. Not required for the exam, but it's the next step in the maturity model and worth knowing exists.

3. Governed tags enforce a vocabulary. Governed tags are defined with acceptable values, thus a tag cannot be set with an incorrect or unaccepted value when attaching it to an object via SQL. In the UI, a dropdown list of the values likewise enforce these values when a user sets a tag. If a user were able to apply any tag value they want, policies then become ineffective, as they have no way of knowing about free form tag values and cannot hold those object to the policy.

4. Both the per-table and ABAC policies are evaluated, and only rows that meet both criteria are displayed. ABAC in this scenario is the stricter policy, and will still filter on top of less strict policies applied to the same object. There's not a "policy precedence".

5. Heavy use of ABAC policies implies a need to balance least-needed group privileges and access with broad scope policies. Nested groups can help with this, but present additional overhead if group hierarchies grow too deep. One strategy is to have a small number of broad, stable groups as policy targets and handle fine-grained membership work at the nested team-level groups beneath the broad role groups. Policies can then remain readable and stable, without having to list a larger number of team groups in its definition, while nested groups can handle membership change without rewriting policy. One trade off is debugging user permissions, as deep group hierarchies make tracing the path of privileges less straight forward. 

